In [0]:
# BROKEN BOOKINGS TABLE TO CHECK THE RULES

from pyspark.sql import functions as F
CATALOG = "airbnb_obs"

# start from real bronze bookings
df = spark.table(f"{CATALOG}.bronze.bookings")

# INJECT 3 problems:
# 1) null out listing_id on ~20 rows  -> should fail "listing_id not null"
# 2) set an illegal booking_status    -> should fail "status valid"
# 3) set nights_booked = 999 on a row -> should fail "nights 1-30"
broken = (df
    .withColumn("listing_id",
        F.when(F.rand(seed=1) < 0.004, F.lit(None)).otherwise(F.col("listing_id")))
    .withColumn("booking_status",
        F.when(F.rand(seed=2) < 0.003, F.lit("PENDING_REVIEW")).otherwise(F.col("booking_status")))
    .withColumn("nights_booked",
        F.when(F.rand(seed=3) < 0.002, F.lit("999")).otherwise(F.col("nights_booked")))
)

broken.write.mode("overwrite").option("overwriteSchema","true") \
      .saveAsTable(f"{CATALOG}.bronze.bookings_broken")

print("✔ created bronze.bookings_broken with injected issues")

In [0]:
# THE RULES
from pyspark.sql import functions as F
from datetime import datetime, timezone
import uuid

CATALOG = "airbnb_obs"

def check_not_null(df, column):
    return df.filter(F.col(column).isNull() | (F.trim(F.col(column)) == "")).count()

def check_unique(df, column):
    dupes = df.groupBy(column).count().filter("count > 1")
    row = dupes.agg(F.sum(F.col("count") - 1).alias("extra")).collect()[0]["extra"]
    return int(row) if row is not None else 0

def check_range(df, column, min_val=None, max_val=None):
    c = F.col(column).cast("double")
    cond = F.lit(False)
    if min_val is not None: cond = cond | (c < min_val)
    if max_val is not None: cond = cond | (c > max_val)
    return df.filter(cond).count()

def check_accepted_values(df, column, allowed):
    return df.filter(~F.col(column).isin(allowed)).count()

def check_row_count_min(df, minimum):
    return 1 if df.count() < minimum else 0

def run_rules(layer, table_name, rules):
    df = spark.table(f"{CATALOG}.{layer}.{table_name}")
    total = df.count()
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:6]
    results = []
    for r in rules:
        rtype = r["rule_type"]
        col   = r.get("column", "*")
        if   rtype == "not_null":        failed = check_not_null(df, col)
        elif rtype == "unique":          failed = check_unique(df, col)
        elif rtype == "range":           failed = check_range(df, col, r.get("min"), r.get("max"))
        elif rtype == "accepted_values": failed = check_accepted_values(df, col, r["allowed"])
        elif rtype == "row_count":       failed = check_row_count_min(df, r["min"])
        else: raise ValueError(f"unknown rule_type {rtype}")

        failed = 0 if failed is None else failed
        results.append((
            run_id, datetime.now(timezone.utc), layer, table_name, col,
            r["rule_name"], rtype, int(failed), int(total),
            "PASS" if failed == 0 else "FAIL", r.get("details", "")
        ))
    cols = ["run_id","check_ts","layer","table_name","column_name","rule_name",
            "rule_type","failed_records","total_records","status","details"]
    spark.createDataFrame(results, cols) \
         .write.mode("append").saveAsTable(f"{CATALOG}.monitoring.dq_results")
    print(f"✔ {table_name}: ran {len(rules)} rules (run_id={run_id})")
    return run_id

In [0]:


#POINT THE SAME RULES TO THE BROKEN TABLE

booking_rules = [
    {"rule_name": "booking_id unique",  "rule_type": "unique",   "column": "booking_id"},
    {"rule_name": "listing_id not null","rule_type": "not_null", "column": "listing_id"},
    {"rule_name": "status valid",       "rule_type": "accepted_values", "column": "booking_status",
     "allowed": ["confirmed", "cancelled"]},
    {"rule_name": "nights 1-30",        "rule_type": "range",    "column": "nights_booked", "min": 1, "max": 30},
]

run_rules("bronze", "bookings_broken", booking_rules)

In [0]:
# VERDICTS

from pyspark.sql import functions as F
display(
    spark.table("airbnb_obs.monitoring.dq_results")
         .filter("table_name = 'bookings_broken'")
         .select("rule_name","status","failed_records","total_records")
         .orderBy(F.desc("check_ts"))
)